Get Spotify data and explore it!

In [ ]:

%load_ext autoreload 
%autoreload 2

import os
import sys
from dotenv import load_dotenv
import logging

sys.path.append('./src')
from api.lyrics_client import LyricsClient
from api.spotify_client import SpotifyClient
from embeddings.embedding_manager import EmbeddingManager # need to downgrade onnxruntime 1.15.1 !!!
from models import SearchQuery
from search.hybrid_search import HybridSearchEngine
from services.lyrics_fetch_service import LyricsFetchService
from services.sync_service import SyncService
from storage.csv_tables import PlaylistsTable, SongsTable, PlaylistTracksTable, LyricsTable
from storage.state_store import StateStore

logger = logging.getLogger(__name__)
logger.setLevel('INFO')

In [ ]:
# Load environment variables
load_dotenv()

# get secrets
client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")
redirect_uri = os.getenv("SPOTIFY_REDIRECT_URI", "http://localhost:3000/callback")

In [ ]:
emb_manager = EmbeddingManager()
search_engine = HybridSearchEngine(emb_manager)
my_sync_service = SyncService()
sp = SpotifyClient()
lc = LyricsClient()
lfs = LyricsFetchService()

In [ ]:
sync_results = my_sync_service.sync_all(spotify=sp, lyrics_client=lc, lyrics_service=lfs, search_engine=search_engine)

In [ ]:
sync_results

In [ ]:
lfs.status()

In [ ]:
len(lfs.lyrics._read_all())/len(lfs.songs._read_all())*100

In [ ]:
# TEMP: create embeddings for all retrieved lyrics

all_lyrics = []

rows = lfs.songs._read_all()
for r in rows:
    sid = r.get('song_id')
    lobj = lfs.lyrics.get_lyrics(sid)
    if lobj != 'lyrics unavailable':

        item = {
            'song_id': sid,
            'title': r.get('title') or '',
            'artist': r.get('artist') or '',
            'album': r.get('album') or '', 
            'duration_ms': str(r.get('duration_ms') or 0),
            'popularity': str(r.get('popularity') or 0),
            'release_date': r.get('release_date') or '', 
            'lyrics_text': lobj.get('lyrics_text') or '',
            'language': lobj.get('language') or '',
            'lyrics_source': lobj.get('source'),
            'lyrics_length': str(len(lobj.get('lyrics_text') or ''))
        }
        all_lyrics.append(item)

emb_manager.add_batch(all_lyrics)

In [ ]:
from models import SearchQuery

search_query = SearchQuery(query='feeling happy', search_type='hybrid', max_results=10)
res = search_engine.search(search_query)
res.results

In [ ]:
#exact search score = 1 if found, 0 otherwise
#fuzzy search score between 0 and 100
#keyword search score (tfdid 0-1, bm25 unbounded)
#semantic search score 0-1